07_Qualitative_Analysis_Gemini.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/07_Qualitative_Analysis_Gemini.ipynb

<a target="_blank" href="https://colab.research.google.com/github/urcraft/data_analytics_lecture_notebooks/blob/main/07_Qualitative_Analysis_Gemini.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Qualitative Data Analysis with Gemini: Thematic Coding

## What you will learn
- Use an LLM to perform first-pass thematic coding on interview data.
- Compare LLM-generated themes against human coding.
- Understand where LLMs help and where they fall short for qualitative research.

**NLP Task**: Qualitative analysis — thematic coding, content analysis.

**Scenario**: NordWind Energy conducted employee interviews about adopting a new
AI-powered maintenance scheduling system. We analyze the transcripts.

**Key limitation**: LLMs work best as *assistants* to human researchers, not replacements.

Expected runtime: 5–10 minutes
Expected cost: Free-tier Gemini


In [ ]:
%pip install -q google-genai pandas


In [ ]:
import os
import json
import time
import pandas as pd

# --- Gemini Setup ---
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'
print('Using Gemini model:', GEMINI_MODEL)

from google import genai

api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('GOOGLE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    raise ValueError('Set GOOGLE_API_KEY environment variable (or Colab secret).')

client = genai.Client(api_key=api_key)
print('Gemini client ready.')


## Sample Data: Interview Excerpts

These are fictional but realistic interview excerpts from NordWind Energy employees
discussing the rollout of an AI maintenance scheduling tool.

Each excerpt has been pre-coded by a human researcher with themes.


In [ ]:
INTERVIEW_DATA = [
    {
        "participant": "P01",
        "role": "Senior Technician",
        "excerpt": "I've been doing this for fifteen years. The AI schedule told me to inspect NW-089 on Tuesday, but anyone with experience knows that site is only accessible at low tide. The system doesn't understand site conditions at all. I ended up ignoring it and doing my usual route.",
        "human_themes": ["local knowledge gap", "trust deficit", "workaround behavior"]
    },
    {
        "participant": "P02",
        "role": "Operations Manager",
        "excerpt": "The dashboard is actually quite useful for the weekly planning meetings. I can see which turbines the system flags as high priority and then discuss with the team whether they agree. It's a good starting point for conversation, not a final answer.",
        "human_themes": ["decision support value", "human-AI collaboration", "management adoption"]
    },
    {
        "participant": "P03",
        "role": "Junior Technician",
        "excerpt": "Honestly? I find it helpful. I don't have 20 years of experience like Lars, so when the system suggests checking the gearbox bearings based on vibration data, that actually teaches me something. It's like having a knowledgeable colleague pointing things out.",
        "human_themes": ["learning tool", "experience gap bridging", "positive reception"]
    },
    {
        "participant": "P04",
        "role": "Site Supervisor",
        "excerpt": "My concern is accountability. If the AI says a turbine is fine and then the gearbox fails, who's responsible? Me? The vendor? The data team in Copenhagen? Nobody's been clear about that, and it makes me nervous about relying on it for safety-critical decisions.",
        "human_themes": ["accountability ambiguity", "safety concern", "organizational governance gap"]
    },
    {
        "participant": "P05",
        "role": "Data Analyst",
        "excerpt": "The model works well when the sensor data is clean, which is maybe 70% of the time. The other 30% — missing readings, sensor drift, the offshore turbines that lose connectivity in storms — that's where it breaks down. Garbage in, garbage out. We need better data infrastructure before we can trust the AI recommendations.",
        "human_themes": ["data quality dependency", "infrastructure gap", "realistic assessment"]
    },
    {
        "participant": "P06",
        "role": "Senior Technician",
        "excerpt": "They rolled this out without asking us. One day there's a new app on our tablets. No training, no explanation of how it works, just 'follow the schedule.' That's not how you build trust with the people who actually do the work. Some of the lads just turn it off.",
        "human_themes": ["change management failure", "lack of consultation", "resistance", "trust deficit"]
    },
    {
        "participant": "P07",
        "role": "HR Manager",
        "excerpt": "I worry about the skills question. If the junior technicians rely on the AI to tell them what to inspect, will they ever develop the intuition that makes Lars so valuable? Are we creating a dependency that makes us more vulnerable, not less?",
        "human_themes": ["skill development concern", "long-term dependency risk", "institutional knowledge"]
    },
    {
        "participant": "P08",
        "role": "Finance Director",
        "excerpt": "The business case is compelling on paper. If we can reduce unplanned downtime by even 15%, that's millions of kroner in saved revenue. But I need to see evidence, not projections. After six months, I want to compare our actual downtime numbers against the predictions.",
        "human_themes": ["ROI focus", "evidence-based evaluation", "measured adoption"]
    },
]

interview_df = pd.DataFrame(INTERVIEW_DATA)
print(f'Loaded {len(interview_df)} interview excerpts from {interview_df["role"].nunique()} different roles.')


## Step 1: LLM-Assisted Thematic Coding (Open Coding)

We ask Gemini to identify themes in each excerpt — mimicking the first pass
of qualitative coding. The goal is to see if the LLM finds similar themes
to the human coder.


In [ ]:
CODING_PROMPT = """You are assisting with qualitative data analysis of an interview about
AI adoption in a wind energy company.

Read the following interview excerpt and identify 2-4 key themes present in the text.
For each theme, provide:
- A short theme label (2-4 words)
- A brief justification (one sentence explaining why this theme is present)

Interview excerpt (Participant {participant}, {role}):
\"\"\"{excerpt}\"\"\"

Respond in this exact JSON format:
{{"themes": [{{"label": "theme label", "justification": "one sentence"}}]}}"""

def code_excerpt(row: dict) -> dict:
    prompt = CODING_PROMPT.format(**row)
    try:
        resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
        raw = resp.text.strip()
        if raw.startswith('```'):
            raw = raw.split('\n', 1)[1] if '\n' in raw else raw[3:]
        if raw.endswith('```'):
            raw = raw.rsplit('```', 1)[0]
        return json.loads(raw.strip())
    except Exception as e:
        return {'themes': [], 'error': str(e)}

results = []
for _, row in interview_df.iterrows():
    result = code_excerpt(row.to_dict())
    results.append(result)
    time.sleep(0.5)

interview_df['llm_themes'] = [
    [t['label'] for t in r.get('themes', [])] for r in results
]
interview_df['llm_justifications'] = [
    [t.get('justification', '') for t in r.get('themes', [])] for r in results
]

print('Coding complete.\n')


## Step 2: Compare Human vs. LLM Themes


In [ ]:
print(f'{"Participant":<6} {"Role":<20} {"Human Themes":<50} {"LLM Themes"}')
print('=' * 130)
for _, row in interview_df.iterrows():
    human = ', '.join(row['human_themes'])
    llm = ', '.join(row['llm_themes'])
    print(f'{row["participant"]:<6} {row["role"]:<20} {human:<50} {llm}')


## Step 3: Cross-Excerpt Theme Aggregation (Axial Coding)

Now we ask the LLM to look across ALL excerpts and identify overarching themes.
This is a higher-order qualitative analysis task.


In [ ]:
all_excerpts = '\n\n'.join([
    f'[{row["participant"]}, {row["role"]}]: "{row["excerpt"]}"'
    for _, row in interview_df.iterrows()
])

AXIAL_PROMPT = f"""You are conducting qualitative data analysis on employee interviews about
AI adoption at a wind energy company.

Here are all interview excerpts:

{all_excerpts}

Based on ALL excerpts together, identify 4-6 overarching themes that emerge across
multiple participants. For each theme:
- Provide a clear theme label
- List which participants (P01-P08) support this theme
- Write a 2-sentence description

Respond in JSON format:
{{"overarching_themes": [{{"label": "...", "participants": ["P01", "P03"], "description": "..."}}]}}"""

try:
    resp = client.models.generate_content(model=GEMINI_MODEL, contents=AXIAL_PROMPT)
    raw = resp.text.strip()
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1] if '\n' in raw else raw[3:]
    if raw.endswith('```'):
        raw = raw.rsplit('```', 1)[0]
    axial_result = json.loads(raw.strip())
except Exception as e:
    axial_result = {'overarching_themes': [], 'error': str(e)}

print('=== Overarching Themes (LLM Axial Coding) ===\n')
for theme in axial_result.get('overarching_themes', []):
    participants = ', '.join(theme.get('participants', []))
    print(f'Theme: {theme["label"]}')
    print(f'  Participants: {participants}')
    print(f'  {theme.get("description", "")}')
    print()


## Discussion Questions

1. **Did the LLM find the same themes as the human coder?**
   Where did it agree? Where did it diverge? Which is "right"?

2. **What did the LLM miss?**
   Cultural nuance, sarcasm, implied meaning — LLMs often produce
   surface-level coding that misses deeper interpretive layers.

3. **Is this useful as a *first pass*?**
   The LLM won't replace a qualitative researcher, but could it
   save time on initial coding before human review?

4. **Data privacy concerns**:
   These are employee interviews about their employer. In a real
   scenario, sending this data to a cloud API raises serious
   GDPR and ethical questions.

## Export


In [ ]:
export_rows = []
for _, row in interview_df.iterrows():
    export_rows.append({
        'participant': row['participant'],
        'role': row['role'],
        'excerpt': row['excerpt'],
        'human_themes': ', '.join(row['human_themes']),
        'llm_themes': ', '.join(row['llm_themes']),
        'student_assessment': '',
    })
export_df = pd.DataFrame(export_rows)
export_path = 'qualitative_coding_results.xlsx'
export_df.to_excel(export_path, index=False)
print(f'Saved {export_path}')

try:
    from google.colab import files
    files.download(export_path)
except Exception:
    print('Download the file from the notebook working directory.')
